# Model selection and experiment tracking

This notebook selects the best candidate model based on cross-validation performance, evaluates it once on the untouched test set and records the experiment with MLflow.

The preprocessing and baseline comparison are available in `02_preprocessing_and_baseline.ipynb`.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/raw/telco_customer_churn.csv")

df = pd.read_csv(DATA_PATH)

TARGET = "Churn"
ID_COLUMN = "customerID"

feature_columns = [
    column
    for column in df.columns
    if column not in [TARGET, ID_COLUMN]
]

X = df[feature_columns].copy()
y = df[TARGET].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

Training rows: 5634
Test rows: 1409


## Reusable preprocessing

The same preprocessing configuration used during baseline evaluation is recreated here. Numeric values are imputed and scaled, while categorical values are imputed and encoded inside a single leakage-safe pipeline.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_preprocessor() -> ColumnTransformer:
    """Create the preprocessing workflow for model inputs."""

    numeric_features = [
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
    ]

    categorical_features = [
        column
        for column in feature_columns
        if column not in numeric_features
    ]

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("one_hot_encoder", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ]
    )

## Correct the numeric type of TotalCharges

`TotalCharges` contains blank strings in the raw file. They are converted to missing numeric values before the preprocessing pipeline imputes them using the training-set median.

In [3]:
def clean_total_charges(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with TotalCharges represented as a numeric column."""

    cleaned = frame.copy()

    cleaned["TotalCharges"] = pd.to_numeric(
        cleaned["TotalCharges"],
        errors="coerce",
    )

    return cleaned


X_train = clean_total_charges(X_train)
X_test = clean_total_charges(X_test)

print(f"Training dtype: {X_train['TotalCharges'].dtype}")
print(f"Missing values in training: {X_train['TotalCharges'].isna().sum()}")
print(f"Missing values in test: {X_test['TotalCharges'].isna().sum()}")

Training dtype: float64
Missing values in training: 8
Missing values in test: 3


## Candidate models

The two candidate models use the same preprocessing logic. This keeps the comparison fair: differences in performance should come from the classifier rather than from different data transformations.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


def build_logistic_regression_model() -> Pipeline:
    """Create the interpretable baseline classification pipeline."""

    return Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1_000,
                    random_state=42,
                ),
            ),
        ]
    )


def build_random_forest_model() -> Pipeline:
    """Create the non-linear random forest classification pipeline."""

    return Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=400,
                    min_samples_leaf=5,
                    max_features="sqrt",
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    )


candidate_models = {
    "logistic_regression": build_logistic_regression_model(),
    "random_forest": build_random_forest_model(),
}

list(candidate_models)

['logistic_regression', 'random_forest']

## Cross-validation experiment tracking with MLflow

MLflow records the model name, configuration and cross-validation metrics for each candidate. The test set remains untouched during this stage.

In [6]:
from pathlib import Path

import mlflow
from sklearn.model_selection import StratifiedKFold, cross_validate

database_path = Path("../mlflow.db").resolve()

mlflow.set_tracking_uri(
    f"sqlite:///{database_path.as_posix()}"
)

mlflow.set_experiment("customer-churn-model-selection")

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "accuracy": "accuracy",
}

2026/09/30 18:01:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/30 18:01:22 INFO mlflow.store.db.utils: Updating database tables
2026/09/30 18:01:23 INFO mlflow.tracking.fluent: Experiment with name 'customer-churn-model-selection' does not exist. Creating a new experiment.


## Compare and log candidate models

Each candidate is evaluated using identical folds and metrics. MLflow stores the resulting parameters and mean cross-validation scores, making the selection decision reproducible.

In [7]:
experiment_results = []

for model_name, model in candidate_models.items():
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
    )

    mean_metrics = {
        metric.replace("test_", ""): values.mean()
        for metric, values in cv_results.items()
        if metric.startswith("test_")
    }

    classifier = model.named_steps["classifier"]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_param("model_name", model_name)
        mlflow.log_params(classifier.get_params())
        mlflow.log_metrics(
            {
                f"cv_{metric}": value
                for metric, value in mean_metrics.items()
            }
        )

    experiment_results.append(
        {
            "model": model_name,
            **mean_metrics,
        }
    )

model_comparison = (
    pd.DataFrame(experiment_results)
    .set_index("model")
    .sort_values("roc_auc", ascending=False)
)

model_comparison.round(3)

,roc_auc,precision,recall,f1,accuracy
model,,,,,
logistic_regression,0.846,0.655,0.543,0.593,0.803
random_forest,0.845,0.540,0.765,0.633,0.764


## Select the candidate model

ROC-AUC is used as the primary selection metric because it measures ranking quality across possible decision thresholds and is less misleading than accuracy for an imbalanced churn dataset.

In [8]:
selection_metric = "roc_auc"

selected_model_name = model_comparison[selection_metric].idxmax()

selected_model = candidate_models[selected_model_name]

print(f"Selected model: {selected_model_name}")
print(
    f"Cross-validation {selection_metric}: "
    f"{model_comparison.loc[selected_model_name, selection_metric]:.3f}"
)

Selected model: logistic_regression
Cross-validation roc_auc: 0.846


## Final holdout evaluation

The selected model is fitted on the complete training set and evaluated once on the untouched test set. These results estimate how well the workflow may generalize to new customers.

In [9]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

selected_model.fit(X_train, y_train)

y_test_prediction = selected_model.predict(X_test)
y_test_probability = selected_model.predict_proba(X_test)[:, 1]

test_metrics = {
    "roc_auc": roc_auc_score(y_test, y_test_probability),
    "precision": precision_score(y_test, y_test_prediction),
    "recall": recall_score(y_test, y_test_prediction),
    "f1": f1_score(y_test, y_test_prediction),
    "accuracy": accuracy_score(y_test, y_test_prediction),
}

pd.Series(test_metrics).sort_values(ascending=False).round(3)

roc_auc      0.842
accuracy     0.806
precision    0.657
f1           0.604
recall       0.559
dtype: float64

## Final evaluation summary

The selected model achieved a test ROC-AUC of **0.842**, close to its cross-validation performance. This suggests that the model generalizes consistently to unseen customers.

At the default decision threshold, the model identifies 55.9% of customers who churn, with a precision of 65.7%. Threshold selection can later be adapted to the business cost of missed churn cases versus unnecessary retention actions.

In [10]:
classifier = selected_model.named_steps["classifier"]

with mlflow.start_run(run_name=f"final_{selected_model_name}"):
    mlflow.log_param("selected_model", selected_model_name)
    mlflow.log_params(classifier.get_params())

    mlflow.log_metrics(
        {
            f"test_{metric}": value
            for metric, value in test_metrics.items()
        }
    )

print("Final evaluation logged in MLflow.")

Final evaluation logged in MLflow.
